In [1]:
import joblib
import os

from pipeline_utils import classify_email, display_result, classify_batch, test_emails

## Load Pre-trained Models

In [2]:
# Define models directory
models_dir = '../models'

# Load Stage 1: Spam Detection Model (Logistic Regression)
spam_model = joblib.load(os.path.join(models_dir, 'lr_model.joblib'))
spam_vectorizer = joblib.load(os.path.join(models_dir, 'lr_vectorizer.joblib'))

# Load Stage 2: Phishing Type Classification Model (Logistic Regression)
phishing_model = joblib.load(os.path.join(models_dir, 'phishing_lr_model.joblib'))
phishing_vectorizer = joblib.load(os.path.join(models_dir, 'phishing_lr_vectorizer.joblib'))

print(f"Spam model classes: {spam_model.classes_}")
print(f"Phishing model classes: {phishing_model.classes_}")

Spam model classes: [0 1]
Phishing model classes: ['authority_scam' 'credential_harvesting' 'financial_scam'
 'generic_phishing' 'legitimate' 'romance_dating' 'social_engineering'
 'social_engineering_advanced' 'tech_support' 'threats' 'urgency']


## Test the Pipeline with Examples

In [3]:
print("EMAIL CLASSIFICATION RESULTS")

for i, email in enumerate(test_emails, 1):
    print(f"\n>>> Email #{i}")
    result = classify_email(email, spam_model, spam_vectorizer, phishing_model, phishing_vectorizer)
    display_result(result)

EMAIL CLASSIFICATION RESULTS

>>> Email #1
📧 EMAIL CLASSIFICATION RESULT

📝 Email Preview:
    Hi John,
    
    Just wanted to follow up on our meeting yesterday. I've attached the quarterly report 
    as discussed. Let me know if you have any questions.
    
    Best regards,
    Sarah

🔍 Stage 1 - Spam Detection:
    Result:     Ham
    Confidence: 98.0%

📊 FINAL CLASSIFICATION:
    Result:     ✅ Legitimate Email (Ham)
    Risk Level: Low

💡 ADVICE:
    This email appears to be legitimate, but always verify sender addresses.


>>> Email #2
📧 EMAIL CLASSIFICATION RESULT

📝 Email Preview:
    URGENT: Your account has been compromised!
    
    We detected suspicious activity on your account. Click the link below to verify 
    your password and secure your account immediately:
    
    [Ve...

🔍 Stage 1 - Spam Detection:
    Result:     Spam
    Confidence: 95.5%

🎯 Stage 2 - Phishing Type Classification:
    Prediction: social_engineering
    Confidence: 31.9%
    2nd Best:   creden

## Batch Classification

In [4]:
batch_results = classify_batch(test_emails, spam_model, spam_vectorizer, phishing_model, phishing_vectorizer)
print("Batch Classification Results:")
batch_results

Batch Classification Results:


,email_preview,spam_detection,spam_confidence,phishing_type,phishing_confidence,alternative_type,alternative_confidence,has_conflict,final_classification,risk_level
0,"Hi John,\n \n Just wanted to follow up o...",Ham,98.0%,N/A,N/A,None,N/A,No,✅ Legitimate Email (Ham),Low
1,URGENT: Your account has been compromised!\n ...,Spam,95.5%,social_engineering,31.9%,credential_harvesting,24.4%,No,🎭 Social Engineering,Medium
2,Congratulations! You have been selected as the...,Spam,97.0%,financial_scam,68.6%,legitimate,14.1%,No,💰 Financial Scam,High
3,FINAL NOTICE: IRS Tax Violation\n \n Thi...,Spam,52.0%,legitimate,34.5%,threats,16.8%,⚠️ Yes,⚠️ ⚠️ Threat/Extortion (Uncertain),Medium-High
4,ALERT: Your computer has been infected!\n \...,Spam,86.2%,legitimate,39.2%,social_engineering,19.3%,⚠️ Yes,⚠️ 🎭 Social Engineering (Uncertain),Medium-High


## Interactive Classification (Enter Your Own Email)

In [5]:
# Try your own email
your_email = """
Storage
Don't risk losing your photos, videos, contacts, files and personal private data.

96%
●	Photos	Full
●	Files	Full
●	Family	Full
●	E-mails	Full
●	Device backup	Almost Full
johndoe24 You may not be able to send or receive emails. To continue using cloud services, please free up space or upgrade your storage. Your files are safe for now, but may be deleted if the cloud is not refreshed.

Upgrade now and get an extra 50 GB bonus storage. Don't wait!
This special offer expires in 4 minutes et 39 seconds

UPDATE
"""

result = classify_email(your_email, spam_model, spam_vectorizer, phishing_model, phishing_vectorizer)
display_result(result)

📧 EMAIL CLASSIFICATION RESULT

📝 Email Preview:
    
Storage
Don't risk losing your photos, videos, contacts, files and personal private data.

96%
●	Photos	Full
●	Files	Full
●	Family	Full
●	E-mails	Full
●	Device backup	Almost Full
johndoe24 You may no...

🔍 Stage 1 - Spam Detection:
    Result:     Spam
    Confidence: 71.2%

🎯 Stage 2 - Phishing Type Classification:
    Prediction: legitimate
    Confidence: 50.9%
    2nd Best:   urgency (10.6%)

🔀 CONFLICT DETECTED
    ⚠️ CONFLICTING PREDICTIONS: Our spam detector flagged this as spam (71.2% confident), but the phishing classifier suggested 'legitimate' (50.9% confident). Showing next most likely phishing type instead.

📊 FINAL CLASSIFICATION:
    Result:     ⚠️ ⏰ Urgency Scam (Uncertain)
    Risk Level: Medium-High

💡 ADVICE:
    Legitimate organizations rarely demand immediate action. Take your time and verify. Note: There is some uncertainty in this classification. Exercise extra caution.

